***Imports***

In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import operator
from pathlib import Path
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.model_selection import TimeSeriesSplit

 ***Carregar o dataset***

In [3]:
csv_base = Path.cwd().parent.parent.parent / "data" 
csv_path_limpo = Path.cwd().parent.parent.parent / "data" / "dataset_completo_limpo.v2.csv"
df = pd.read_csv(csv_path_limpo)

In [13]:
csv_path_limpoI = csv_base / "dataset_completo.v1.csv"

In [14]:
dfII = pd.read_csv(csv_path_limpoI)

In [15]:
dfII.tail()

,YEAR,QUARTER,MONTH,DAY_OF_MONTH,DAY_OF_WEEK,FL_DATE,OP_UNIQUE_CARRIER,ORIGIN_AIRPORT_ID,ORIGIN_AIRPORT_SEQ_ID,ORIGIN_CITY_MARKET_ID,...,DEP_DELAY_NEW,DEP_DEL15,DEP_DELAY_GROUP,CRS_ARR_TIME,ARR_TIME,ARR_DELAY,ARR_DELAY_NEW,ARR_DEL15,CANCELLED,DIVERTED
13577019,2023,4,12,31,7,12/31/2023 12:00:00 AM,YX,15096,1509602,35096,...,0.0,0.0,-1.0,1314,1238.0,-36.0,0.0,0.0,0.0,0.0
13577020,2023,4,12,31,7,12/31/2023 12:00:00 AM,YX,15096,1509602,35096,...,0.0,0.0,-1.0,733,741.0,8.0,8.0,0.0,0.0,0.0
13577021,2023,4,12,31,7,12/31/2023 12:00:00 AM,YX,15096,1509602,35096,...,25.0,1.0,1.0,1855,1918.0,23.0,23.0,1.0,0.0,0.0
13577022,2023,4,12,31,7,12/31/2023 12:00:00 AM,YX,15370,1537002,34653,...,0.0,0.0,-1.0,1507,1448.0,-19.0,0.0,0.0,0.0,0.0
13577023,2023,4,12,31,7,12/31/2023 12:00:00 AM,YX,15919,1591905,31834,...,0.0,0.0,-1.0,1045,1049.0,4.0,4.0,0.0,0.0,0.0


In [9]:
df.tail()

,Year,Quarter,Month,DayofMonth,DayOfWeek,Reporting_Airline,OriginAirportID,OriginCityMarketID,OriginStateFips,DestAirportID,...,DestStateFips,CRSElapsedTime,Distance,DistanceGroup,CRSDepHour,CRSDepMinute,CRSArrHour,CRSArrMinute,Target,MilesPerMinute
13275412,2023,3,9,1,5,YX,12264,30852,51,14122,...,42,67,182,1,12,53,14,0,1,2.716418
13275413,2023,3,9,1,5,YX,11618,31703,34,10431,...,37,136,583,3,20,0,22,16,1,4.286765
13275414,2023,3,9,1,5,YX,11618,31703,34,14730,...,21,145,642,3,16,29,18,54,1,4.427586
13275415,2023,3,9,1,5,YX,11618,31703,34,11433,...,26,119,488,2,13,59,15,58,1,4.100840
13275416,2023,3,9,1,5,YX,10994,30994,45,12264,...,51,95,441,2,14,50,16,25,1,4.642105


In [5]:
df.isnull().sum()

Year                  0
Quarter               0
Month                 0
DayofMonth            0
DayOfWeek             0
Reporting_Airline     0
OriginAirportID       0
OriginCityMarketID    0
OriginStateFips       0
DestAirportID         0
DestCityMarketID      0
DestStateFips         0
CRSElapsedTime        0
Distance              0
DistanceGroup         0
CRSDepHour            0
CRSDepMinute          0
CRSArrHour            0
CRSArrMinute          0
Target                0
dtype: int64

**Split dos dados**

In [4]:
y = df['Target']
X = df.drop(columns=['Target'])

In [38]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42,stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42,stratify=y_train)

In [8]:
len(X_train[X_train['CRSElapsedTime']<=0])

5

In [39]:
mask = X_train["CRSElapsedTime"] > 0

X_train = X_train.loc[mask]
y_train = y_train.loc[mask]

Retirar coluna do ano

In [40]:
X_train.drop(columns=['Year'],inplace=True)
X_val.drop(columns=['Year'],inplace=True)
X_test.drop(columns=['Year'],inplace=True)

In [42]:
X_train.columns

Index(['Quarter', 'Month', 'DayofMonth', 'DayOfWeek', 'Reporting_Airline',
       'OriginAirportID', 'OriginCityMarketID', 'OriginStateFips',
       'DestAirportID', 'DestCityMarketID', 'DestStateFips', 'CRSElapsedTime',
       'Distance', 'DistanceGroup', 'CRSDepHour', 'CRSDepMinute', 'CRSArrHour',
       'CRSArrMinute', 'MilesPerMinute'],
      dtype='object')

In [5]:
colunas_numericas = [
    'Quarter', 'Month', 'DayofMonth', 'DayOfWeek', 'CRSElapsedTime',
    'Distance', 'DistanceGroup', 'CRSDepHour', 'CRSDepMinute',
    'CRSArrHour', 'CRSArrMinute', 'MilesPerMinute'
]

colunas_categoricas = [
    'OriginAirportID', 'DestAirportID', 'Reporting_Airline', 
    'OriginCityMarketID', 'DestCityMarketID', 'OriginStateFips', 'DestStateFips'
]

ct = ColumnTransformer([
    ('num', StandardScaler(), colunas_numericas),
    ('cat', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), colunas_categoricas)
], remainder='drop')

ct.set_output(transform="pandas")


#X_train = pd.DataFrame(X_train, columns=X_train.columns)
X_train = pd.DataFrame(ct.fit_transform(X_train))
X_val = ct.transform(X_val)
X_test = ct.transform(X_test)

NameError: name 'X_train' is not defined

In [32]:
X_train.head()

,num__Quarter,num__Month,num__DayofMonth,num__DayOfWeek,num__CRSElapsedTime,num__Distance,num__DistanceGroup,num__CRSDepHour,num__CRSDepMinute,num__CRSArrHour,num__CRSArrMinute,cat__OriginAirportID,cat__DestAirportID,cat__Reporting_Airline,cat__OriginCityMarketID,cat__DestCityMarketID,cat__OriginStateFips,cat__DestStateFips
5320172,0.424576,0.120636,1.171013,-0.988408,-0.431831,-0.393202,-0.329313,-0.208486,-1.431851,-0.299692,1.379961,264.0,75.0,1.0,266.0,89.0,8.0,32.0
11709425,0.424576,0.120636,1.056875,-0.988408,1.518002,1.841261,1.801990,-2.649231,-0.443046,-1.448392,0.248158,316.0,106.0,4.0,44.0,109.0,46.0,21.0
735736,1.328058,1.002895,-0.198635,0.509566,-0.830037,-0.682328,-0.755573,-1.225463,-1.486785,-1.256942,-0.317743,325.0,308.0,11.0,192.0,250.0,4.0,4.0
211469,-1.382388,-1.643883,0.486189,0.010241,-0.294519,-0.222734,-0.329313,-0.208486,0.161224,-0.108241,0.191568,97.0,77.0,8.0,16.0,91.0,42.0,12.0
1914184,1.328058,1.591068,1.399287,-0.489083,0.652935,0.686427,0.523208,-0.208486,1.424698,0.466109,-1.392956,114.0,256.0,11.0,50.0,81.0,42.0,12.0


In [7]:
X.tail()

,Year,Quarter,Month,DayofMonth,DayOfWeek,Reporting_Airline,OriginAirportID,OriginCityMarketID,OriginStateFips,DestAirportID,DestCityMarketID,DestStateFips,CRSElapsedTime,Distance,DistanceGroup,CRSDepHour,CRSDepMinute,CRSArrHour,CRSArrMinute,MilesPerMinute
13275412,2023,3,9,1,5,YX,12264,30852,51,14122,30198,42,67,182,1,12,53,14,0,2.716418
13275413,2023,3,9,1,5,YX,11618,31703,34,10431,30431,37,136,583,3,20,0,22,16,4.286765
13275414,2023,3,9,1,5,YX,11618,31703,34,14730,33044,21,145,642,3,16,29,18,54,4.427586
13275415,2023,3,9,1,5,YX,11618,31703,34,11433,31295,26,119,488,2,13,59,15,58,4.100840
13275416,2023,3,9,1,5,YX,10994,30994,45,12264,30852,51,95,441,2,14,50,16,25,4.642105


In [22]:
X = X.sort_values(by=['Year', 'Month', 'DayofMonth']).reset_index(drop=True)

# Se y estiver separado, ordene y com a mesma ordem
y = y.loc[X.index].reset_index(drop=True)

In [23]:
tscv = TimeSeriesSplit(n_splits=4)

for fold, (train_index, val_index) in enumerate(tscv.split(X)):
    print(f"\n--- Avaliando FOLD {fold + 1} ---")

    train_data = X.iloc[train_index]
    val_data   = X.iloc[val_index]

    # Início e Fim do Treino (usando .iloc na coluna)
    day_tI    = train_data['DayofMonth'].iloc[0]
    month_tI  = train_data['Month'].iloc[0]
    year_tI   = train_data['Year'].iloc[0]

    day_tII   = train_data['DayofMonth'].iloc[-1]
    month_tII = train_data['Month'].iloc[-1]
    year_tII  = train_data['Year'].iloc[-1]

    print(f"Training date:   {day_tI}/{month_tI}/{year_tI} até {day_tII}/{month_tII}/{year_tII}")

    # Início e Fim da Validação (usando .iloc na coluna)
    day_vI    = val_data['DayofMonth'].iloc[0]
    month_vI  = val_data['Month'].iloc[0]
    year_vI   = val_data['Year'].iloc[0]

    day_vII   = val_data['DayofMonth'].iloc[-1]
    month_vII = val_data['Month'].iloc[-1]
    year_vII  = val_data['Year'].iloc[-1]

    print(f"Validation date: {day_vI}/{month_vI}/{year_vI} até {day_vII}/{month_vII}/{year_vII}")


--- Avaliando FOLD 1 ---
Training date:   1/1/2022 até 1/6/2022
Validation date: 1/6/2022 até 23/10/2022

--- Avaliando FOLD 2 ---
Training date:   1/1/2022 até 23/10/2022
Validation date: 23/10/2022 até 23/3/2023

--- Avaliando FOLD 3 ---
Training date:   1/1/2022 até 23/3/2023
Validation date: 23/3/2023 até 12/8/2023

--- Avaliando FOLD 4 ---
Training date:   1/1/2022 até 12/8/2023
Validation date: 12/8/2023 até 31/12/2023


In [20]:
print("Última linha de X:")
print(f"{X['DayofMonth'].iloc[-1]}/{X['Month'].iloc[-1]}/{X['Year'].iloc[-1]}")

Última linha de X:
1/9/2023


In [21]:
print(X.groupby('Year')['Month'].value_counts().sort_index())

Year  Month
2022  1        503529
      2        473146
      3        554710
      4        542117
      5        565955
      6        557753
      7        582549
      8        573619
      9        547728
      10       566492
      11       539199
      12       525216
2023  1        527197
      2        492747
      3        571533
      4        550249
      5        575430
      6        562804
      7        585058
      8        592142
      9        560887
      10       596003
      11       562413
      12       566941
Name: count, dtype: int64
